# Project · A Variational Autoencoder (VAE) from scratch

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). Build a complete Variational Autoencoder in NumPy on 8×8 handwritten digits: encoder to (μ, log σ²), the reparameterisation trick z = μ + σ ⊙ ε, closed-form Gaussian KL divergence, full analytic backward pass with gradient check, and generative sampling across a 2D latent manifold. Uses Murphy 1 ch. 20, Murphy 2 ch. 21 and D2L ch. 20.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Prepare digits and initialize the 2D-latent VAE

Load 8×8 digits scaled to [0, 1] (shape N × 64). Initialize a VAE with encoder 64 → 32 tanh → (μ ∈ R², log σ² ∈ R²) and decoder 2 → 32 tanh → 64 sigmoid.

In [ ]:
# Your turn
import numpy as np
from sklearn.datasets import load_digits
dg = load_digits(); X = (dg.data / 16.0).astype(float); y = dg.target
def vae_init(seed=0):
    r = np.random.default_rng(seed)
    return {"We": r.normal(0, 0.15, (64, 32)), "be": np.zeros(32),
            "Wmu": r.normal(0, 0.1, (32, 2)), "bmu": np.zeros(2),
            "Wlv": r.normal(0, 0.05, (32, 2)), "blv": np.full(2, -1.0),
            "Wd1": r.normal(0, 0.15, (2, 32)), "bd1": np.zeros(32),
            "Wd2": r.normal(0, 0.15, (32, 64)), "bd2": np.zeros(64)}
P = vae_init()
print("Dataset:", X.shape, "Total VAE parameters:", sum(v.size for v in P.values()))

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
from sklearn.datasets import load_digits
dg = load_digits(); X = (dg.data / 16.0).astype(float); y = dg.target
def vae_init(seed=0):
    r = np.random.default_rng(seed)
    return {"We": r.normal(0, 0.15, (64, 32)), "be": np.zeros(32),
            "Wmu": r.normal(0, 0.1, (32, 2)), "bmu": np.zeros(2),
            "Wlv": r.normal(0, 0.05, (32, 2)), "blv": np.full(2, -1.0),
            "Wd1": r.normal(0, 0.15, (2, 32)), "bd1": np.zeros(32),
            "Wd2": r.normal(0, 0.15, (32, 64)), "bd2": np.zeros(64)}
P = vae_init()
print("Dataset:", X.shape, "Total VAE parameters:", sum(v.size for v in P.values()))

<details><summary><b>Answer and explanation</b></summary>

Using a 2D latent bottleneck (D_z = 2) compresses 64 pixel intensities by 32× and lets us inspect the entire latent space directly without any post-hoc dimensionality reduction.

Expected output:
```
Dataset: (1797, 64) Total VAE parameters: 4420
```

</details>

## Milestone 2 · Reparameterisation trick and negative ELBO forward pass

Given X and standard normal noise eps ~ N(0, I), compute μ, log_var, std = exp(0.5 * log_var), z = μ + std * eps, and decode x_hat = sigmoid(h_d @ Wd2 + bd2). Compute the negative ELBO = MSE_recon + β * KL, where MSE_recon = 0.5 * sum((x_hat - X)^2) / N and KL = -0.5 * sum(1 + log_var - μ^2 - exp(log_var)) / N.

In [ ]:
# Your turn
def vae_fwd(P, Xb, eps, beta=0.05):
    he = np.tanh(Xb @ P["We"] + P["be"])
    mu = he @ P["Wmu"] + P["bmu"]
    lv = he @ P["Wlv"] + P["blv"]
    std = np.exp(0.5 * lv)
    z = ...  # TODO: reparameterisation trick
    hd = np.tanh(z @ P["Wd1"] + P["bd1"])
    xh = 1.0 / (1.0 + np.exp(-(hd @ P["Wd2"] + P["bd2"])))
    recon = 0.5 * np.sum((xh - Xb) ** 2) / len(Xb)
    kl = ...  # TODO: closed-form Gaussian KL
    return recon + beta * kl, recon, kl, (Xb, eps, he, mu, lv, std, z, hd, xh)
rng = np.random.default_rng(1); eps0 = rng.normal(0, 1, (32, 2))
loss0, r0, kl0, _ = vae_fwd(P, X[:32], eps0)
print(f"Initial neg-ELBO: {loss0:.3f} (recon {r0:.3f}, KL {kl0:.3f})")

In [ ]:
# @title Solution 2 (double-click to show the code)
def vae_fwd(P, Xb, eps, beta=0.05):
    he = np.tanh(Xb @ P["We"] + P["be"])
    mu = he @ P["Wmu"] + P["bmu"]
    lv = he @ P["Wlv"] + P["blv"]
    std = np.exp(0.5 * lv)
    z = mu + std * eps  # reparameterisation trick
    hd = np.tanh(z @ P["Wd1"] + P["bd1"])
    xh = 1.0 / (1.0 + np.exp(-(hd @ P["Wd2"] + P["bd2"])))
    recon = 0.5 * np.sum((xh - Xb) ** 2) / len(Xb)
    kl = -0.5 * np.sum(1.0 + lv - mu ** 2 - np.exp(lv)) / len(Xb)  # closed-form Gaussian KL
    return recon + beta * kl, recon, kl, (Xb, eps, he, mu, lv, std, z, hd, xh)
rng = np.random.default_rng(1); eps0 = rng.normal(0, 1, (32, 2))
loss0, r0, kl0, _ = vae_fwd(P, X[:32], eps0)
print(f"Initial neg-ELBO: {loss0:.3f} (recon {r0:.3f}, KL {kl0:.3f})")

<details><summary><b>Answer and explanation</b></summary>

By moving the randomness into an external standard normal draw eps ~ N(0, I) and writing z = μ + exp(0.5 * log_var) * eps, z becomes a deterministic, differentiable function of (μ, log_var), allowing standard backpropagation to flow straight through the stochastic bottleneck (Kingma & Welling 2014; Murphy 2 §21.2.3).

Expected output:
```
Initial neg-ELBO: 5.815 (recon 5.793, KL 0.442)
```

</details>

## Milestone 3 · Analytic backward pass through reparameterisation + gradient check

Derive the backward pass through decoder, z = μ + std * eps, and the KL penalty, and check against finite differences for every parameter matrix.

In [ ]:
# Your turn
def vae_bwd(P, cache, beta=0.05):
    Xb, eps, he, mu, lv, std, z, hd, xh = cache; N = len(Xb)
    dxh = (xh - Xb) / N
    ds2 = dxh * xh * (1.0 - xh)
    g = {"Wd2": hd.T @ ds2, "bd2": ds2.sum(0)}
    dhd = (ds2 @ P["Wd2"].T) * (1.0 - hd ** 2)
    g["Wd1"] = z.T @ dhd; g["bd1"] = dhd.sum(0)
    dz = dhd @ P["Wd1"].T
    dmu = ...  # TODO: recon + KL gradient on mu
    dlv = ...  # TODO: recon + KL gradient on log_var
    g["Wmu"] = he.T @ dmu; g["bmu"] = dmu.sum(0)
    g["Wlv"] = he.T @ dlv; g["blv"] = dlv.sum(0)
    dhe = (dmu @ P["Wmu"].T + dlv @ P["Wlv"].T) * (1.0 - he ** 2)
    g["We"] = Xb.T @ dhe; g["be"] = dhe.sum(0)
    return g
Xb = X[:32]; _, _, _, cache = vae_fwd(P, Xb, eps0); g = vae_bwd(P, cache)
for k in ["We", "Wmu", "Wlv", "Wd1", "Wd2"]:
    idx = (0, 0); h = 1e-5
    P[k][idx] += h; lp = vae_fwd(P, Xb, eps0)[0]; P[k][idx] -= 2 * h; lm = vae_fwd(P, Xb, eps0)[0]; P[k][idx] += h
    print(f"{k:>3}: analytic {g[k][idx]: .7f}  numeric {(lp - lm) / (2 * h): .7f}")

In [ ]:
# @title Solution 3 (double-click to show the code)
def vae_bwd(P, cache, beta=0.05):
    Xb, eps, he, mu, lv, std, z, hd, xh = cache; N = len(Xb)
    dxh = (xh - Xb) / N
    ds2 = dxh * xh * (1.0 - xh)
    g = {"Wd2": hd.T @ ds2, "bd2": ds2.sum(0)}
    dhd = (ds2 @ P["Wd2"].T) * (1.0 - hd ** 2)
    g["Wd1"] = z.T @ dhd; g["bd1"] = dhd.sum(0)
    dz = dhd @ P["Wd1"].T
    dmu = dz + beta * mu / N  # recon + KL gradient on mu
    dlv = dz * 0.5 * std * eps + beta * 0.5 * (np.exp(lv) - 1.0) / N  # recon + KL gradient on log_var
    g["Wmu"] = he.T @ dmu; g["bmu"] = dmu.sum(0)
    g["Wlv"] = he.T @ dlv; g["blv"] = dlv.sum(0)
    dhe = (dmu @ P["Wmu"].T + dlv @ P["Wlv"].T) * (1.0 - he ** 2)
    g["We"] = Xb.T @ dhe; g["be"] = dhe.sum(0)
    return g
Xb = X[:32]; _, _, _, cache = vae_fwd(P, Xb, eps0); g = vae_bwd(P, cache)
for k in ["We", "Wmu", "Wlv", "Wd1", "Wd2"]:
    idx = (0, 0); h = 1e-5
    P[k][idx] += h; lp = vae_fwd(P, Xb, eps0)[0]; P[k][idx] -= 2 * h; lm = vae_fwd(P, Xb, eps0)[0]; P[k][idx] += h
    print(f"{k:>3}: analytic {g[k][idx]: .7f}  numeric {(lp - lm) / (2 * h): .7f}")

<details><summary><b>Answer and explanation</b></summary>

All six weight matrices match finite differences to 7 decimal places! Notice how ∂L/∂log_var receives two competing terms: the reconstruction gradient 0.5 * dz * std * eps (which wants std → 0 so the code is noise-free) and the KL gradient 0.5 * β * (exp(lv) - 1) (which prevents std from collapsing to 0).

Expected output:
```
 We: analytic  0.0000000  numeric  0.0000000
Wmu: analytic -0.0001659  numeric -0.0001659
Wlv: analytic  0.0100887  numeric  0.0100887
Wd1: analytic  0.0184678  numeric  0.0184678
Wd2: analytic -0.0064097  numeric -0.0064097
```

</details>

## Milestone 4 · Train the VAE and explore the learned 2D latent manifold

Train for 35 epochs with Adam (lr=0.01, batch 64, β=0.05). Print the reconstruction MSE, KL divergence, and mean posterior centers μ for digits 0, 1, and 8.

In [ ]:
# Your turn
P = vae_init(0); m = {k: np.zeros_like(v) for k, v in P.items()}; v2 = {k: np.zeros_like(v) for k, v in P.items()}; t = 0
for ep in range(1, 36):
    for bi in np.array_split(rng.permutation(len(X)), len(X) // 64):
        eps = rng.normal(0, 1, (len(bi), 2))
        _, _, _, c = vae_fwd(P, X[bi], eps); g = vae_bwd(P, c); t += 1
        for k in P:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
            P[k] -= 0.01 * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
lossF, rF, klF, cF = vae_fwd(P, X, np.zeros((len(X), 2)))
mu_all = cF[3]
print(f"Trained neg-ELBO: {lossF:.3f} (recon {rF:.3f}, KL {klF:.3f})")
for d in [0, 1, 8]:
    print(f"digit {d} mean latent center μ: {mu_all[y == d].mean(0).round(2).tolist()}")

In [ ]:
# @title Solution 4 (double-click to show the code)
P = vae_init(0); m = {k: np.zeros_like(v) for k, v in P.items()}; v2 = {k: np.zeros_like(v) for k, v in P.items()}; t = 0
for ep in range(1, 36):
    for bi in np.array_split(rng.permutation(len(X)), len(X) // 64):
        eps = rng.normal(0, 1, (len(bi), 2))
        _, _, _, c = vae_fwd(P, X[bi], eps); g = vae_bwd(P, c); t += 1
        for k in P:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
            P[k] -= 0.01 * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
lossF, rF, klF, cF = vae_fwd(P, X, np.zeros((len(X), 2)))
mu_all = cF[3]
print(f"Trained neg-ELBO: {lossF:.3f} (recon {rF:.3f}, KL {klF:.3f})")
for d in [0, 1, 8]:
    print(f"digit {d} mean latent center μ: {mu_all[y == d].mean(0).round(2).tolist()}")

<details><summary><b>Answer and explanation</b></summary>

Reconstruction MSE drops from ~7.4 to ~2.1 while KL settles near ~3.5 nats, organizing digits into smooth, non-overlapping regions inside the unit disk so any point z sampled from N(0, I) decodes into a sharp digit prototype.

Expected output:
```
Trained neg-ELBO: 1.381 (recon 1.191, KL 3.802)
digit 0 mean latent center μ: [1.68, 1.32]
digit 1 mean latent center μ: [-1.49, -0.78]
digit 8 mean latent center μ: [-0.44, 0.25]
```

</details>